# Task 1 — Fine-tune Qwen2.5-Coder-7B-Instruct

Notebook này chạy **QLoRA 4-bit** trên đúng 26 bài train và đánh giá trên 6 bài validation. Chạy `01_prepare_data.ipynb` trước để tạo `task1_train.jsonl`, `task1_val.jsonl` và split trên Google Drive. Chọn runtime **GPU** trong Colab. Dữ liệu và adapter chỉ lưu trong Drive cá nhân; không commit bài nộp sinh viên hoặc checkpoint.

Quy trình: cài thư viện → mount Drive → kiểm tra dữ liệu/token → smoke test 1 bước → train 3 epoch → xem checkpoint tốt nhất.

In [ ]:
%pip -q install "transformers>=4.46,<5" "peft>=0.13,<1" "accelerate>=0.34,<2" "bitsandbytes>=0.45,<1" "pyyaml>=6" "pydantic>=2.5"

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))

PROJECT_HOME = Path('/content/drive/MyDrive/LLM/Final Project')
REPO_CANDIDATES = [PROJECT_HOME / 'llm-final-project', Path('/content/llm-final-project')]
REPO_DIR = next((p for p in REPO_CANDIDATES if (p / 'scripts/train_task1.py').is_file()), None)
assert REPO_DIR is not None, 'Đồng bộ repo mới lên Drive (hoặc /content/llm-final-project) rồi chạy lại.'
PROCESSED_DIR = PROJECT_HOME / 'data/processed'
SPLITS_DIR = PROJECT_HOME / 'data/splits'
OUTPUT_DIR = PROJECT_HOME / 'data/outputs/task1/qwen25_coder7b_qlora'
required = [PROCESSED_DIR / 'task1_train.jsonl', PROCESSED_DIR / 'task1_val.jsonl', SPLITS_DIR / 'train_ids.json', SPLITS_DIR / 'val_ids.json']
missing = [str(p) for p in required if not p.is_file()]
assert not missing, 'Thiếu dữ liệu đã xử lý. Chạy colab/01_prepare_data.ipynb trước: ' + str(missing)
print('Repo:', REPO_DIR)
print('Data:', PROCESSED_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
RUNNER = REPO_DIR / 'scripts/train_task1.py'
CONFIG = REPO_DIR / 'configs/task1/lora_7b.yaml'

def run_task1(*flags):
    command = [sys.executable, str(RUNNER), '--config', str(CONFIG),
               '--processed-dir', str(PROCESSED_DIR), '--splits-dir', str(SPLITS_DIR),
               '--output-dir', str(OUTPUT_DIR), *flags]
    subprocess.run(command, cwd=REPO_DIR, check=True)

run_task1('--preflight')

## Smoke test và huấn luyện đầy đủ

Smoke test dùng một optimizer step và một bài validation, lưu trong thư mục `smoke/`. Lượt đầy đủ dùng cả 26/6 bài, lưu checkpoint mỗi epoch. Nếu Colab mất kết nối giữa chừng, kiểm tra thư mục output trước khi chạy lại để tránh ghi đè kết quả cần giữ.

In [ ]:
RUN_SMOKE = True
if RUN_SMOKE:
    run_task1('--smoke')
    smoke_summary = json.loads((OUTPUT_DIR / 'smoke/training_summary.json').read_text(encoding='utf-8'))
    print('Smoke test OK. QWK:', smoke_summary['selected_metrics']['qwk'])

In [ ]:
RUN_FULL = True
if RUN_FULL:
    run_task1()
    summary = json.loads((OUTPUT_DIR / 'training_summary.json').read_text(encoding='utf-8'))
    print('Best epoch:', summary['selected_epoch'])
    print('Validation QWK:', summary['selected_metrics']['qwk'])
    print('Validation MAE:', summary['selected_metrics']['mae'])
    print('Valid output rate:', summary['selected_metrics']['valid_output_rate'])
    print('Adapter:', OUTPUT_DIR / 'best_adapter')
    print('All metrics:', OUTPUT_DIR / 'training_summary.json')